# 05 低评分订单风险预测

这一阶段把前面的业务发现转成机器学习问题：预测订单是否会收到低评分。

## 业务问题

能否在订单完成后、评价产生前，识别更可能收到低评分的订单？

这里把 `review_score <= 2` 定义为低评分订单。这个模型适合用于客服跟进、物流复盘、卖家治理和客户体验预警。

## 建模原则

- 明确定义标签：`is_low_review`
- 避免数据泄露：不使用任何 review score、review text、review count 等评价结果字段作为特征
- 使用时间切分：用较早订单训练，用较晚订单测试，更接近真实业务上线场景
- 不只看 accuracy：低评分是少数类，要重点看 ROC-AUC、PR-AUC、precision、recall、F1
- 保留可解释性：先用 Logistic Regression 建一个能解释方向的模型


## 0. 导入依赖与路径


In [ ]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
matplotlib_cache_dir = project_root / ".matplotlib_cache"
matplotlib_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(matplotlib_cache_dir)

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.4f}".format
plt.style.use("ggplot")

processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

processed_dir, reports_dir, figures_dir


## 1. 读取分析主表


In [ ]:
orders = pd.read_csv(processed_dir / "orders_analysis_base.csv")
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

for col in ["is_delivered", "is_late", "is_low_review"]:
    orders[col] = orders[col].replace({"True": True, "False": False}).astype("boolean")

orders.shape


## 2. 定义建模样本和标签

这里使用已送达、已有评价、支付金额不为空的订单。

注意：这不是全量订单预测模型，而是一个“已完成订单的低评分风险识别模型”。如果业务目标改成下单时预测风险，需要重新设计特征，不能使用实际配送时长。

In [ ]:
model_data = orders[
    orders["is_delivered"].fillna(False)
    & orders["is_low_review"].notna()
    & orders["payment_total"].notna()
    & orders["order_purchase_timestamp"].notna()
].copy()

model_data["target_low_review"] = model_data["is_low_review"].astype(int)
model_data["is_late_num"] = model_data["is_late"].fillna(False).astype(int)

target_summary = model_data["target_low_review"].value_counts(normalize=False).rename_axis("target_low_review").reset_index(name="orders")
target_summary["share"] = target_summary["orders"] / target_summary["orders"].sum()
target_summary


## 3. 特征设计

排除所有评价结果字段：

- `review_score_mean`
- `review_score_min`
- `review_score_max`
- `review_count`
- `has_review_title`
- `has_review_message`
- `has_review`
- `is_low_review`

保留订单、支付、商品、物流、地区和品类特征。

In [ ]:
numeric_features = [
    "payment_total",
    "product_total",
    "freight_total",
    "freight_share_of_payment",
    "item_count",
    "product_count",
    "seller_count",
    "product_category_count",
    "avg_item_price",
    "avg_freight_ratio",
    "payment_count",
    "payment_type_count",
    "max_payment_installments",
    "approval_time_hours",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delta_days",
    "is_late_num",
]

categorical_features = [
    "purchase_month",
    "purchase_dayofweek",
    "customer_state",
    "main_product_category",
    "main_seller_state",
    "primary_payment_type",
]

feature_cols = numeric_features + categorical_features

leakage_cols = [col for col in feature_cols if "review" in col.lower()]
assert not leakage_cols, f"Potential leakage columns found: {leakage_cols}"

model_data[feature_cols].head()


## 4. 时间切分训练集和测试集

用 2018-01-01 之前的订单训练，用 2018-01-01 及之后的订单测试。这样比随机切分更接近真实业务：模型只能从过去学习，然后预测未来。

In [ ]:
cutoff_date = pd.Timestamp("2018-01-01")

train_df = model_data[model_data["order_purchase_timestamp"] < cutoff_date].copy()
test_df = model_data[model_data["order_purchase_timestamp"] >= cutoff_date].copy()

X_train = train_df[feature_cols]
y_train = train_df["target_low_review"]
X_test = test_df[feature_cols]
y_test = test_df["target_low_review"]

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "low_review_rate": [y_train.mean(), y_test.mean()],
    "start_date": [train_df["order_purchase_timestamp"].min(), test_df["order_purchase_timestamp"].min()],
    "end_date": [train_df["order_purchase_timestamp"].max(), test_df["order_purchase_timestamp"].max()],
})

split_summary


## 5. 建立预处理和模型

这里训练两个模型：

- `DummyClassifier`：无业务价值的基线模型，用来提醒自己 accuracy 容易误导
- `LogisticRegression`：可解释、稳定、适合第一版业务风险模型


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

dummy_model = DummyClassifier(strategy="prior")

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
])


## 6. 训练和评估


In [ ]:
def evaluate_classifier(name, model, X_train, y_train, X_test, y_test, threshold=0.5):
    model.fit(X_train, y_train)
    y_score = model.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= threshold).astype(int)
    return {
        "model": name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }, y_score

dummy_metrics, dummy_scores = evaluate_classifier("dummy_prior", dummy_model, X_train, y_train, X_test, y_test)
logistic_metrics, logistic_scores = evaluate_classifier("logistic_regression", logistic_model, X_train, y_train, X_test, y_test)

metrics_df = pd.DataFrame([dummy_metrics, logistic_metrics])
metrics_df


## 7. 阈值选择

默认阈值 0.5 不一定适合风险预警。这里选择 F1 最高的阈值，平衡 precision 和 recall。


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, logistic_scores)
f1_values = np.divide(
    2 * precision[:-1] * recall[:-1],
    precision[:-1] + recall[:-1],
    out=np.zeros_like(precision[:-1]),
    where=(precision[:-1] + recall[:-1]) != 0,
)

best_idx = int(np.nanargmax(f1_values))
best_threshold = float(thresholds[best_idx])

tuned_metrics, tuned_scores = evaluate_classifier(
    "logistic_regression_tuned_threshold",
    logistic_model,
    X_train,
    y_train,
    X_test,
    y_test,
    threshold=best_threshold,
)

metrics_df = pd.DataFrame([dummy_metrics, logistic_metrics, tuned_metrics])
metrics_df


## 8. 模型可视化


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, logistic_scores, ax=ax, name="Logistic Regression")
ax.set_title("Low Review Risk Model - ROC Curve")
plt.tight_layout()
plt.savefig(figures_dir / "model_low_review_roc_curve.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, logistic_scores, ax=ax, name="Logistic Regression")
ax.set_title("Low Review Risk Model - Precision Recall Curve")
plt.tight_layout()
plt.savefig(figures_dir / "model_low_review_precision_recall_curve.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
tuned_pred = (logistic_scores >= best_threshold).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, tuned_pred, ax=ax, values_format=",d")
ax.set_title(f"Confusion Matrix at Threshold {best_threshold:.3f}")
plt.tight_layout()
plt.savefig(figures_dir / "model_low_review_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

confusion_matrix(y_test, tuned_pred)


## 9. 模型解释

Logistic Regression 的系数可以解释为风险方向：

- 正系数：更容易预测为低评分
- 负系数：更不容易预测为低评分

注意：系数不是因果关系，只是模型在当前特征下学习到的关联。

In [ ]:
trained_preprocessor = logistic_model.named_steps["preprocessor"]
classifier = logistic_model.named_steps["classifier"]

feature_names = trained_preprocessor.get_feature_names_out()
coef = classifier.coef_[0]

feature_importance = (
    pd.DataFrame({"feature": feature_names, "coefficient": coef})
    .assign(abs_coefficient=lambda df: df["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

feature_importance.head(20)


In [ ]:
top_positive = feature_importance.sort_values("coefficient", ascending=False).head(12)
top_negative = feature_importance.sort_values("coefficient", ascending=True).head(12)
plot_importance = pd.concat([top_negative, top_positive]).sort_values("coefficient")

fig, ax = plt.subplots(figsize=(11, 8))
colors = ["#4c78a8" if value < 0 else "#d62728" for value in plot_importance["coefficient"]]
ax.barh(plot_importance["feature"], plot_importance["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Most Influential Logistic Regression Coefficients")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.savefig(figures_dir / "model_low_review_feature_coefficients.png", dpi=160, bbox_inches="tight")
plt.show()


## 10. 保存结果


In [ ]:
metrics_path = reports_dir / "model_low_review_metrics.csv"
feature_importance_path = reports_dir / "model_low_review_feature_importance.csv"
summary_path = reports_dir / "modeling_low_review_summary.md"

metrics_df.to_csv(metrics_path, index=False)
feature_importance.to_csv(feature_importance_path, index=False)

champion = metrics_df.loc[metrics_df["model"].eq("logistic_regression_tuned_threshold")].iloc[0]
baseline = metrics_df.loc[metrics_df["model"].eq("dummy_prior")].iloc[0]

summary = f"""# Low Review Risk Modeling Summary

## Objective

Predict whether a delivered order is likely to receive a low review score (`review_score <= 2`).

## Data Split

- Training period: orders before 2018-01-01
- Test period: orders on or after 2018-01-01
- Training rows: {len(train_df):,}
- Test rows: {len(test_df):,}
- Test low-review rate: {y_test.mean():.1%}

## Leakage Control

The model excludes review text, review score, review count, and any field derived directly from the review outcome.

## Model Performance

| Model | Threshold | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Dummy prior | {baseline['threshold']:.3f} | {baseline['roc_auc']:.3f} | {baseline['pr_auc']:.3f} | {baseline['precision']:.3f} | {baseline['recall']:.3f} | {baseline['f1']:.3f} |
| Logistic tuned | {champion['threshold']:.3f} | {champion['roc_auc']:.3f} | {champion['pr_auc']:.3f} | {champion['precision']:.3f} | {champion['recall']:.3f} | {champion['f1']:.3f} |

## Interpretation

I treat the model as a risk-ranking tool rather than an automated decision system. It can help prioritize orders for customer support follow-up or operational review.

A useful next improvement would be to compare two versions of the model: one available at purchase time and one available after delivery. That would separate prevention use cases from post-delivery recovery use cases.
"""

summary_path.write_text(summary, encoding="utf-8")

print(summary)
print(f"Saved metrics to: {metrics_path}")
print(f"Saved feature importance to: {feature_importance_path}")
print(f"Saved summary to: {summary_path}")


## 11. 本阶段结论

本阶段完成了一个完整的机器学习小模块：

- 明确业务目标
- 定义标签
- 控制数据泄露
- 做时间切分
- 建立 baseline 和可解释模型
- 用适合不平衡分类的指标评估
- 输出模型图表和总结报告

后续可以继续整理 Dashboard 或 README。Dashboard 用于展示业务指标；README 用于说明项目方法、关键发现和结果。